# BBDM: Planck 100 GHz -> ACT+Planck 90 GHz

Суперразрешение карт реликтового излучения диффузионной моделью на
броуновском мосте.

**Что важно помнить при работе с этим ноутбуком:**

- Направление моста: `t = 0 -> y` (ACT+Planck, цель), `t = T -> x0` (Planck, вход).
  В коде `x0` означает "патч Planck", а не "состояние модели в t=0".
- В сэмплер попадает **только** Planck. Целевая карта на инференсе не
  передаётся никуда и ни в каком виде.
- Любая спектральная метрика считается **по >= 30-60 патчам**. Одиночный
  патч на высоких ell шумит так, что систематика и шум оценки визуально
  неотличимы.
- Изменение `BBDM.loss` / `BBDM.q_sample` требует полного переобучения.
  Изменение `BBDM.sample` / `_posterior_coeffs` -- нет, достаточно
  пересэмплировать существующий чекпоинт (так делаются абляции по ETA и S).

### 0. Окружение (Colab)

In [ ]:
# Colab: подключить Drive и достать репозиторий.
# Локально этот блок не нужен.

# from google.colab import drive
# drive.mount("/content/drive")
# !pip install -q astropy scikit-image tqdm
# !git clone https://github.com/Perf0rator4/bbdm-cmb.git /content/bbdm_cmb

print("Skipped — running from a local checkout")

### 1. Импорты, пути, устройство

In [ ]:
import os
import sys

# Корень репозитория -- папка, в которой лежит пакет bbdm.
for _candidate in (os.getcwd(), os.path.abspath(".."),
                   "/content/bbdm_cmb", "/content/bbdm-cmb"):
    if os.path.isdir(os.path.join(_candidate, "bbdm", "model")):
        REPO_ROOT = _candidate
        break
else:
    raise RuntimeError("Пакет bbdm не найден — задайте REPO_ROOT вручную")

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt

from bbdm.config import (
    PATCH_SIZE, TRAIN_RATIO, VAL_RATIO, SEED,
    MAX_ZERO_FRAC, MAX_MASK_MISMATCH_FRAC,
    IN_CH, BASE_CH, TIME_DIM, GROUPS,
    T, S, S_VAR, ETA,
    COND_ON_INPUT, UPSAMPLE,
    N_EPOCHS, BATCH_SIZE, LR, NUM_WORKERS, SPECTRAL_WEIGHT,
    EMA_START, EMA_DECAY,
    MONITOR_EVERY, MONITOR_N, MONITOR_S,
)
from bbdm.data import CMBPatchDataset, compute_normalization, get_tile_splits
from bbdm.model import BBDM, UNet
from bbdm.train import train, load_checkpoint
from bbdm.sample import run_inference, visualize_inference
from bbdm.evaluate import (
    evaluate_image_metrics,
    evaluate_spectra,
    plot_spectral_comparison,
    print_band_table,
)

# Пути к данным. Либо пропишите их здесь, либо в bbdm/config.py --
# все функции ниже принимают их явными аргументами.
PLANCK_DIR     = "data/Diffusion/Planck/f100/"
ACT_DIR        = "data/Diffusion/Planck+ACT/f090/"
# Отдельная папка на каждый прогон: train() откажется писать туда, где уже
# лежит best.pt/last.pt, чтобы не затереть прошлый многочасовой прогон.
# Чекпоинт run 4 (ETA=0 + spectral loss) оставьте там, где он лежит, и
# открывайте его через load_checkpoint(...) -- архитектура определится сама.
CHECKPOINT_DIR = "checkpoint/run5_mse_cond_resizeconv/"

# На A40 (40 ГБ) помещается 32; на меньшей карте уменьшите.
BATCH_SIZE = 4

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Repo root: {REPO_ROOT}")
print(f"Device:    {device}")
if device == "cuda":
    print(f"GPU:       {torch.cuda.get_device_name(0)}")
    print(f"VRAM:      {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"\nETA             = {ETA}  (0 — как в статье BBDM: x_T = вход ровно)")
print(f"SPECTRAL_WEIGHT = {SPECTRAL_WEIGHT}  (0 — чистый MSE)")
print(f"COND_ON_INPUT   = {COND_ON_INPUT}  (Planck подаётся денойзеру отдельным каналом)")
print(f"UPSAMPLE        = {UPSAMPLE}")

### 2. Разбиение train/val/test

Разбиение делается **на уровне тайлов**, до нарезки на патчи: четыре патча
одного тайла соседние по небу, и разбиение на уровне патчей протащило бы
почти одинаковое небо из train в test.

In [ ]:
train_tiles, val_tiles, test_tiles = get_tile_splits(
    planck_dir=PLANCK_DIR,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    seed=SEED,
)

### 3. Нормализация

Общие (mu, sigma) по ненулевым пикселям **только train-тайлов**, одни и те
же для Planck и ACT: предсказание и цель денормализуются одной парой, и в
Transfer Function нормировка сокращается.

In [ ]:
mu, sigma = compute_normalization(
    planck_dir=PLANCK_DIR,
    act_dir=ACT_DIR,
    train_tiles=train_tiles,
    patch_size=PATCH_SIZE,
)
print(f"mu    = {mu:.4f} uK")
print(f"sigma = {sigma:.4f} uK")

### 4. Датасеты

Патч выбрасывается, если у него слишком много замаскированных пикселей
(`MAX_ZERO_FRAC`) **или** если маски Planck и ACT заметно не совпадают
(`MAX_MASK_MISMATCH_FRAC`) -- у инструментов разные футпринты, и на части
патчей у одного из них срезан угол. Смотрите на печать "dropped ... by
Planck/ACT mask mismatch": если отсеивается заметная доля данных, порог
стоит поднять осознанно, а не молча.

In [ ]:
common = dict(
    planck_dir=PLANCK_DIR,
    act_dir=ACT_DIR,
    mu=mu, sigma=sigma,
    patch_size=PATCH_SIZE,
    max_zero_frac=MAX_ZERO_FRAC,
    max_mask_mismatch_frac=MAX_MASK_MISMATCH_FRAC,
)

train_ds = CMBPatchDataset(tile_list=train_tiles, augment=True,  **common)
val_ds   = CMBPatchDataset(tile_list=val_tiles,   augment=False, **common)
test_ds  = CMBPatchDataset(tile_list=test_tiles,  augment=False, **common)

print(f"\nTrain: {len(train_ds)} samples ({len(train_ds.pairs)} unique pairs)")
print(f"Val:   {len(val_ds)} samples")
print(f"Test:  {len(test_ds)} samples")

x0, y = train_ds[0]
print(f"Patch shape: x0={tuple(x0.shape)}, y={tuple(y.shape)}")

### 5. Sanity check: как выглядят пары

In [ ]:
n_aug = len(train_ds.augmentations)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for col in range(4):
    # Шаг n_aug — чтобы взять 4 РАЗНЫХ патча, а не один в 4 аугментациях.
    x0, y = train_ds[col * n_aug]
    x0_np = train_ds.denormalize(x0[0].numpy())
    y_np  = train_ds.denormalize(y[0].numpy())

    # Общая шкала по цели: при индивидуальной автошкале разница в
    # амплитуде между входом и целью визуально исчезает.
    valid = y_np[y_np != 0]
    vmin, vmax = np.percentile(valid, [2, 98])

    for row, (data, label) in enumerate([(x0_np, "Planck"), (y_np, "ACT+Planck")]):
        ax = axes[row, col]
        im = ax.imshow(data, cmap="RdBu_r", vmin=vmin, vmax=vmax, origin="lower")
        ax.set_xticks([]); ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(label, fontsize=12)

plt.suptitle("Dataset sanity check (общая шкала в каждой колонке)", fontsize=13)
plt.tight_layout()
plt.show()

### 6. Модель (run 5)Три изменения относительно run 4, у каждого своя диагностическая подпись,поэтому их вклад можно разделить по результатам одного прогона:- **`SPECTRAL_WEIGHT = 0`** (чистый MSE). Денойзер обязан выдавать  апостериорное *среднее*; недостающую дисперсию добавляет сэмплер. Run 4  требовал от среднего мощности реализации, и сеть перестала гасить шум в  середине цепочки. Проверка: TF на высоких ell вернулась к уровню run 3  (~0.7–1), а не 6–9.- **Planck отдельным каналом** (`COND_ON_INPUT`). Шаг сэмплера точен,  только если `pred = E[y | x_t, x0]`; сеть, видящая лишь `x_t`, переносит  шум Planck на выход. Проверка: `r_in` (корреляция выхода со входом) в  полосе 0.1–0.3 падает.- **`Upsample + Conv3x3` вместо `ConvTranspose2d`** (`UPSAMPLE`). Проверка:  отношение ось/диагональ у предсказания в полосе 0.3–1.0 возвращается к ~1.

In [ ]:
unet = UNet(
    in_ch=IN_CH, base_ch=BASE_CH, time_dim=TIME_DIM, groups=GROUPS,
    cond_ch=1 if COND_ON_INPUT else 0,
    upsample=UPSAMPLE,
)

bbdm = BBDM(
    model=unet,
    T=T,
    s=S_VAR,
    eta=ETA,                          # 0.0
    spectral_weight=SPECTRAL_WEIGHT,  # 0.0
)

n_params = sum(p.numel() for p in unet.parameters())
print(f"UNet parameters: {n_params:,}")
print(f"UNet size:       {n_params * 4 / 1024**2:.1f} MB")

### 7. Sanity check: один батч

In [ ]:
bbdm = bbdm.to(device)

x0_test, y_test = train_ds[0]
x0_test = x0_test.unsqueeze(0).to(device)
y_test  = y_test.unsqueeze(0).to(device)

with torch.no_grad():
    total, terms = bbdm.loss(x0_test, y_test, return_terms=True)

print(f"loss  = {total.item():.6f}")
print(f"  mse  = {terms['mse'].item():.6f}")
print(f"  spec = {terms['spec'].item():.6f}")
if device == "cuda":
    print(f"VRAM used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print("Forward pass OK")

### 8. Обучение`resume=True` подхватит `last.pt`, если сессия оборвалась. Валидационныйлосс считается с фиксированным сидом, поэтому val-кривая сравнима междуэпохами.**Монитор.** Каждую эпоху на 8 val-патчах прогоняется настоящий сэмплер(S=50) и печатается строка `[monitor ep N] ... TF .. r .. r_in ..` пополосам. Run 4 показал, что одношаговый лосс провал цепочки не видитвообще: val-лосс был в порядке, а TF на выходе — 9.5. Если в первыхэпохах TF в верхних полосах уходит за 2 (монитор это пометит) — стоитостановиться и посмотреть, а не ждать 32 часа.

In [ ]:
bbdm, ema = train(
    bbdm=bbdm,
    train_dataset=train_ds,
    val_dataset=val_ds,
    n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    ema_start=EMA_START,
    spectral_weight=SPECTRAL_WEIGHT,
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    num_workers=NUM_WORKERS,
    resume=False,
    monitor_dataset=val_ds,
    monitor_every=MONITOR_EVERY,
    monitor_n=MONITOR_N,
    monitor_S=MONITOR_S,
)

### 9. Загрузка лучшего чекпоинта`load_checkpoint` сам определяет архитектуру по весам (апсемплинг, каналусловия, T, s) и подставляет EMA-веса. Им же открываются старые чекпоинтыrun 3/4 — например, чтобы прогнать на run 4 новые колонки (`r_in`, 2D-спектрвхода): `bbdm, checkpoint = load_checkpoint("<путь к run 4>/best.pt", device)`.

In [ ]:
bbdm, checkpoint = load_checkpoint(
    os.path.join(CHECKPOINT_DIR, "best.pt"), device=device,
)

print(f"Epoch:    {checkpoint['epoch'] + 1}")
print(f"Val loss: {checkpoint['val_loss']:.6f} (mse {checkpoint['val_mse']:.6f})")
print(f"Hparams:  {checkpoint['hparams']}")
print(f"Arch:     cond_ch={bbdm.model.cond_ch}, upsample={bbdm.model.upsample}")

if checkpoint.get("monitor"):
    last = checkpoint["monitor"][-1]
    print(f"Monitor (epoch {last['epoch'] + 1}, S={last['S']}): TF по полосам "
          + ", ".join(f"{v:.2f}" for v in last["tf_bands"]))

### 10. Инференс: три стохастических реализации

Три сэмпла из одного и того же Planck-патча. Они обязаны различаться:
задача принципиально один-ко-многим, и одинаковые сэмплы означали бы, что
модель схлопнулась в условное среднее.

In [ ]:
x0_val, y_val = val_ds[0]
x0_np = val_ds.denormalize(x0_val[0].numpy())
y_np  = val_ds.denormalize(y_val[0].numpy())

samples = run_inference(
    bbdm=bbdm,
    x0_norm=x0_val.numpy(),   # только Planck; цель сюда не передаётся
    mu=mu, sigma=sigma,
    S=S,
    device=device,
    n_samples=3,
    seed=0,
    progress=True,
)

visualize_inference(x0_np, y_np, samples)

spread = np.std([s for s in samples], axis=0).mean()
print(f"Средний разброс между сэмплами: {spread:.3f} uK "
      f"(0 означало бы схлопывание в условное среднее)")

### 11. Спектральная оценка: Transfer Function и r_ell

Обе величины усредняются по 50 патчам, по схеме "среднее спектров, потом
отношение". Среднее поштучных отношений здесь не годится: если у одного
патча мощность цели в каком-то бине близка к нулю, его отношение уносит
среднее.

TF ~ 1 сам по себе ничего не доказывает. Смотреть надо вместе с `r_ell`:
если лишняя мощность на высоких ell не коррелирует с целью (белый шум,
галлюцинированные точечные источники), TF может быть близка к 1 при
проседающей `r_ell` -- это не физическая точность, а совпадение.

In [ ]:
N_EVAL = 50
eval_indices = list(range(min(N_EVAL, len(val_ds))))

res = evaluate_spectra(
    bbdm=bbdm,
    dataset=val_ds,
    mu=mu, sigma=sigma,
    indices=eval_indices,
    S=S,
    device=device,
    batch_size=4,
)

print_band_table(res, "run 5: MSE + cond + resize-conv")
plot_spectral_comparison([res], ["run 5"])

### 12. Проверка направленных артефактов апсемплингаСтарая архитектура (`ConvTranspose2d(k=2, s=2)`) давала анизотропныйартефакт ровно на частоте Найквиста: разные веса на каждой из четырёхсубпиксельных фаз способны выдать узор с периодом 2 пикселя. Надиагностике run 4 это было видно как яркие пятна на 2D-спектре и какотношение мощности вдоль осей к диагоналям 3.9 у предсказания против 1.09у цели в полосе 0.3–1.0. `UPSAMPLE = "resize_conv"` (nearest + общий conv3x3) от фазы не зависит и должен эту анизотропию убрать.Колонка `input` в таблице ниже -- контроль: если далеко от 1 не только`pred`, но и вход, дело не в сети, а в самих данных (например, в следерепроекции Planck).

In [ ]:
from bbdm.evaluate import compute_2d_power_spectrum, plot_2d_power_spectrum, print_anisotropy_table

spec2d = compute_2d_power_spectrum(
    bbdm, val_ds, n_patches=16, batch_size=4, device=device, S=S,
)
plot_2d_power_spectrum(spec2d, title="run 5")
print_anisotropy_table(spec2d)

### 13. PSNR / SSIM

Приводятся для сопоставимости с литературой. SSIM для стохастической
генерации один-ко-многим -- слабая метрика: она штрафует любую реализацию,
отличную от конкретной наблюдённой, даже при идеальной статистике.
Основной критерий здесь -- TF и r_ell выше.

In [ ]:
metrics = evaluate_image_metrics(
    bbdm=bbdm,
    dataset=val_ds,
    mu=mu, sigma=sigma,
    indices=eval_indices,
    S=S,
    device=device,
    batch_size=4,
)

print(f"PSNR: {metrics['psnr_mean']:.2f} ± {metrics['psnr_std']:.2f} dB")
print(f"SSIM: {metrics['ssim_mean']:.4f} ± {metrics['ssim_std']:.4f}")
print(f"(по {metrics['n_patches']} патчам)")

### 14. Тестовый сплит: визуальная проверка

In [ ]:
rng = np.random.default_rng(SEED)
indices = rng.choice(len(test_ds), size=min(3, len(test_ds)), replace=False)

for i in indices:
    x0_t, y_t = test_ds[int(i)]
    x0_np = test_ds.denormalize(x0_t[0].numpy())
    y_np  = test_ds.denormalize(y_t[0].numpy())

    samples = run_inference(
        bbdm=bbdm,
        x0_norm=x0_t.numpy(),
        mu=mu, sigma=sigma,
        S=S, device=device, n_samples=3, seed=int(i),
    )
    visualize_inference(x0_np, y_np, samples)

### 15. Спектральная оценка на тесте

Финальная цифра для статьи. Валидационный сплит использовался при выборе
чекпоинта и подборе `spectral_weight`, поэтому итоговые TF и r_ell надо
приводить по тесту.

In [ ]:
res_test = evaluate_spectra(
    bbdm=bbdm,
    dataset=test_ds,
    mu=mu, sigma=sigma,
    n_patches=min(60, len(test_ds)),
    S=S,
    device=device,
    batch_size=4,
)

print_band_table(res_test, "TEST")
plot_spectral_comparison([res_test], ["test"], title="Test split")